# Projeto Final - Pré-processamento de Dados em Vendas

## Parte 1 - Ingestão, Diagnóstico e Limpeza

Esta etapa documenta a preparação da base bruta, o diagnóstico de qualidade e os tratamentos aplicados para transformar os dados em uma base confiável, rastreável e pronta para a análise da Parte 2.


## Contexto inicial do projeto

Este notebook concentra a primeira metade do trabalho: entender a origem dos dados, avaliar a qualidade da base bruta e preparar um conjunto consistente para a etapa analítica seguinte.

A lógica aqui é simples, mas essencial: antes de qualquer análise executiva, precisamos garantir que a base represente bem o problema de negócio e que os dados tenham coerência estrutural e semântica.


## Estrutura do Projeto

A entrega foi organizada para seguir uma lógica de projeto de ponta a ponta:
- definição do problema de negócio;
- ingestão e inspeção dos dados;
- análise exploratória inicial;
- diagnóstico de qualidade;
- limpeza e integração;
- consolidação da base tratada;
- análise e interpretação dos resultados.


## Definição do Problema de Negócio

**Pergunta central**
- Como preparar uma base confiável para analisar vendas, mix de produtos e comportamento das movimentações?

**Desafio de negócio**
- Os dados chegam em formatos diferentes e podem conter inconsistências estruturais e operacionais, o que dificulta uma leitura correta do desempenho comercial.

**Objetivo técnico**
- Construir uma base consistente, rastreável e analiticamente útil para sustentar a análise exploratéria e a geração de insights.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.vendas_pipeline import (
    load_movimentacoes,
    load_produtos,
    cast_types_movimentacoes,
    cast_types_produtos,
    merge_base,
    clean_base,
    save_outputs,
)

sns.set_theme(style='whitegrid')

BASE_DIR = Path('..').resolve()
RAW_DIR = BASE_DIR / 'dados' / 'raw'
PROCESSED_DIR = BASE_DIR / 'dados' / 'processed'

df_mov = load_movimentacoes(RAW_DIR)
df_prod = load_produtos(RAW_DIR)


## O que será validado nesta etapa

Ao longo desta parte do projeto, observamos quatro frentes principais de verificação: estrutura das colunas, tipos de dados, presença de inconsistências e aderência dos registros ao caso de negócio.

Essas validações evitam interpretações precipitadas e ajudam a definir quais dados podem seguir para a etapa analítica sem comprometer a leitura dos resultados.


## Pré-Processamento dos Dados

Nesta etapa o projeto inspeciona a estrutura dos arquivos, os tipos de dados, o comportamento inicial das colunas e os primeiros sinais de qualidade da base. O foco aqui é estabelecer uma leitura confiável antes de qualquer análise mais sofisticada.


### Extração e Carregamento

Nesta seção os arquivos de origem são carregados e organizados para permitir inspeção, validação de esquema e comparação entre as fontes disponíveis.


In [2]:
display(df_mov.head(10))
display(df_prod.head(10))

print('DimensÃµes movimentações:', df_mov.shape)
print('DimensÃµes produtos/serviÃ§os:', df_prod.shape)
print('Tipos de dados - movimentações:')
display(df_mov.dtypes)
print('Tipos de dados - produtos/serviÃ§os:')
display(df_prod.dtypes)


,data_emissao,id_produto_servico,id_produto_servico_empresa,tipo_transacao,direcao_estoque,cd_modelo,descricao_modelo,cd_modelo_fiscal,id_pessoa,cd_situacao,...,status_item_cancelado,valor_desconto_digitado,valor_desconto_proporcional,valor_frete_item,cd_cfop,descricao_cfop,qtd_item_movimentacao,qtd_venda,valor_unitario,_arquivo_origem
0,2026-05-01,662875b00345e118a4611764,66731dde0345e118a4611790,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,66731ddf0345e118a46131bd,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,1.592,1.592,99.9,movimentacoes_20260501.json
1,2026-05-01,68adf63a4211dc2f3475d917,68adf63a4211dc2f3475d91c,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,66731ddf0345e118a46131bd,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,1.0,1.0,81.9,movimentacoes_20260501.json
2,2026-05-01,6052c2300345e118a461223e,66731dde0345e118a4612268,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,69e3e0234211dc2e8c649ed4,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,1.0,1.0,42.9,movimentacoes_20260501.json
3,2026-05-01,67e456138df90b3394089f54,67e456138df90b3394089f59,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,69e3e0234211dc2e8c649ed4,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,0.574,0.574,109.9,movimentacoes_20260501.json
4,2026-05-01,686428e94211dc655073d519,686428e94211dc655073d51e,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,69e3e0234211dc2e8c649ed4,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,0.566,0.566,119.9,movimentacoes_20260501.json
5,2026-05-01,698f5ee94211dc233cbbc0e8,698f5ee94211dc233cbbc0ed,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,69e3e0234211dc2e8c649ed4,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,0.776,0.776,139.9,movimentacoes_20260501.json
6,2026-05-01,68e7b4ea4211dc4ad495d4ea,68e7b4ea4211dc4ad495d4ef,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,69e3e0234211dc2e8c649ed4,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,1.0,1.0,13.9,movimentacoes_20260501.json
7,2026-05-01,61230f300345e118a4611b78,66731dde0345e118a4611ba7,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,69e3e0234211dc2e8c649ed4,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,1.0,1.0,13.9,movimentacoes_20260501.json
8,2026-05-01,647aacb00345e118a4611f91,66731dde0345e118a4612000,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,66731ddf0345e118a46130e9,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,0.538,0.538,59.9,movimentacoes_20260501.json
9,2026-05-01,632143b00345e118a4611e6a,66731dde0345e118a4611e8d,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,66731ddf0345e118a4613261,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,0.944,0.944,59.9,movimentacoes_20260501.json


,id_produto_servico,cd_produto_servico,descricao,status_produto_servico,pesavel,vendavel,percentual_cashback,unidade_sigla,tipo_item_descricao,sub_grupo_referencia,...,id_produto_servico_empresa_referencia,id_empresa_referencia,id_estoque_referencia,id_preco_referencia,margem_lucro_aplicada_referencia,limite_desconto_referencia,percentual_comissao_referencia,_source_ingestion_timestamp,_source_file,flag_item_ativo
0,6864293a4211dc655073d54b,1169,Moída 1a FS,True,True,True,0.0,Quilograma,Mercadoria para Revenda,669a6f868df90b307cb5296b,...,6864293a4211dc655073d550,66731a400345e149c87d17a7,6864293a4211dc655073d54c,6864293a4211dc655073d54d,0.0,0.0,0.0,2026-04-18 13:00:08.737143 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True
1,696145824211dc55b032af50,1366,Chuleta c alcatra Frig. Madu,True,True,True,0.0,Quilograma,Mercadoria para Revenda,669a6f868df90b307cb5296b,...,696145824211dc55b032af55,66731a400345e149c87d17a7,696145824211dc55b032af51,696145824211dc55b032af52,0.0,0.0,0.0,2026-04-18 13:00:08.803276 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True
2,685401e94211dc04e86f59d6,1143,Entrecot Las Lilas,False,True,True,0.0,Quilograma,Mercadoria para Revenda,669a6f868df90b307cb5296b,...,685401e94211dc04e86f59db,66731a400345e149c87d17a7,685401e94211dc04e86f59d7,685401e94211dc04e86f59d8,0.0,0.0,0.0,2026-04-18 13:00:08.735199 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,False
3,61b171300345e118a461214d,235,Bife de Alcatra,False,True,True,0.0,Quilograma,Mercadoria para Revenda,669a6f868df90b307cb5296b,...,66731dde0345e118a4612183,66731a400345e149c87d17a7,66731dde0345e118a4612157,66731dde0345e118a4612180,0.0,0.0,0.0,2026-04-18 13:00:08.444784 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,False
4,686426434211dc655073d472,1154,Bife Contrafilé FS,True,True,True,0.0,Quilograma,Mercadoria para Revenda,669a6f868df90b307cb5296b,...,686426434211dc655073d477,66731a400345e149c87d17a7,686426434211dc655073d473,686426434211dc655073d474,0.0,0.0,0.0,2026-04-18 13:00:08.736141 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True
5,62578e300345e118a461268a,265,Picanha Comesul A,False,True,True,0.0,Quilograma,Mercadoria para Revenda,669a6f868df90b307cb5296b,...,66731ddf0345e118a46126c2,66731a400345e149c87d17a7,66731dde0345e118a46126a4,66731ddf0345e118a46126bf,0.0,0.0,0.0,2026-04-18 13:00:08.446591 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,False
6,664eb1300345e118a4612469,753,Vinho Concentus Blend,True,True,True,0.0,Unidade,Mercadoria para Revenda,667c0cdd8df90b4a40c4e1bb,...,66731dde0345e118a461248c,66731a400345e149c87d17a7,66731dde0345e118a461246e,66731dde0345e118a4612489,0.0,0.0,0.0,2026-04-18 13:00:08.530840 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True
7,6610bab00345e118a4611c06,722,Vila Francioni Cabernet Franc,False,True,True,0.0,Unidade,Mercadoria para Revenda,667c0cdd8df90b4a40c4e1bb,...,66731dde0345e118a4611c34,66731a400345e149c87d17a7,66731dde0345e118a4611c1a,66731dde0345e118a4611c31,0.0,0.0,0.0,2026-04-18 13:00:08.528955 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,False
8,66d9cd028df90b091c3b849b,856,Vinho Octopoda Cab. Sauvignon,True,True,True,0.0,Unidade,Mercadoria para Revenda,667c0cdd8df90b4a40c4e1bb,...,66d9cd028df90b091c3b84a0,66731a400345e149c87d17a7,66d9cd028df90b091c3b849c,66d9cd028df90b091c3b849d,0.0,0.0,0.0,2026-04-18 13:00:08.616818 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True
9,620f0bb00345e118a4612514,251,Linguiça Suína Apimentada - DT,False,True,True,0.0,Unidade,Mercadoria para Revenda,669a6f868df90b307cb52970,...,66731dde0345e118a461253f,66731a400345e149c87d17a7,66731dde0345e118a461251f,66731dde0345e118a461253b,0.0,0.0,0.0,2026-04-18 13:00:08.445735 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,False


DimensÃµes movimentações: (708, 21)
DimensÃµes produtos/serviÃ§os: (1389, 22)
Tipos de dados - movimentações:


data_emissao                   object
id_produto_servico             object
id_produto_servico_empresa     object
tipo_transacao                 object
direcao_estoque                object
cd_modelo                      object
descricao_modelo               object
cd_modelo_fiscal               object
id_pessoa                      object
cd_situacao                    object
descricao_situacao             object
status_item_cancelado          object
valor_desconto_digitado        object
valor_desconto_proporcional    object
valor_frete_item               object
cd_cfop                        object
descricao_cfop                 object
qtd_item_movimentacao          object
qtd_venda                      object
valor_unitario                 object
_arquivo_origem                object
dtype: object

Tipos de dados - produtos/serviÃ§os:


id_produto_servico                        object
cd_produto_servico                        object
descricao                                 object
status_produto_servico                      bool
pesavel                                   object
vendavel                                  object
percentual_cashback                      float64
unidade_sigla                             object
tipo_item_descricao                       object
sub_grupo_referencia                      object
codigo_barras                            float64
codigo_barras_tributavel                 float64
id_produto_servico_empresa_referencia     object
id_empresa_referencia                     object
id_estoque_referencia                     object
id_preco_referencia                       object
margem_lucro_aplicada_referencia         float64
limite_desconto_referencia               float64
percentual_comissao_referencia           float64
_source_ingestion_timestamp               object
_source_file        

### Estatística Descritiva

A estatística descritiva fornece uma visão inicial da distribuição dos dados, ajuda a identificar assimetrias e apoia a leitura de possíveis anomalias antes da limpeza.


In [3]:
print('Resumo estatéstico - movimentações')
display(df_mov.describe(include="all"))

print('Resumo estatéstico - produtos/serviÃ§os')
display(df_prod.describe(include="all"))


Resumo estatéstico - movimentações


,data_emissao,id_produto_servico,id_produto_servico_empresa,tipo_transacao,direcao_estoque,cd_modelo,descricao_modelo,cd_modelo_fiscal,id_pessoa,cd_situacao,...,status_item_cancelado,valor_desconto_digitado,valor_desconto_proporcional,valor_frete_item,cd_cfop,descricao_cfop,qtd_item_movimentacao,qtd_venda,valor_unitario,_arquivo_origem
count,708,708,708,708,708,708,708,708,708,708,...,708,708,708,708,708,708,708,708,708,708
unique,11,149,149,3,2,3,3,3,121,2,...,1,12,88,1,4,4,322,322,124,12
top,2026-05-10,67e456138df90b3394089f54,67e456138df90b3394089f59,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,59257eeac1546a18c0360ef3,2,...,false,0.0,0.0,0.0,5102,Venda de mercadoria adquirida ou recebida de t...,1.0,1.0,109.9,movimentacoes_20260510.json
freq,160,49,49,675,676,619,619,619,75,705,...,708,697,608,708,622,622,219,219,61,160


Resumo estatéstico - produtos/serviÃ§os


,id_produto_servico,cd_produto_servico,descricao,status_produto_servico,pesavel,vendavel,percentual_cashback,unidade_sigla,tipo_item_descricao,sub_grupo_referencia,...,id_produto_servico_empresa_referencia,id_empresa_referencia,id_estoque_referencia,id_preco_referencia,margem_lucro_aplicada_referencia,limite_desconto_referencia,percentual_comissao_referencia,_source_ingestion_timestamp,_source_file,flag_item_ativo
count,1389,1389,1389,1389,1388,1360,1389.0,1389,1389,1389,...,1388,1388,1387,1388,1388.0,1388.0,1388.0,1389,1389,1389
unique,1389,1389,1378,2,2,2,NaN,5,3,27,...,1388,1,1387,1388,NaN,NaN,NaN,1389,9,2
top,6864293a4211dc655073d54b,1169,Entrecot FS,True,True,True,NaN,Unidade,Mercadoria para Revenda,669a6f868df90b307cb5296b,...,6864293a4211dc655073d550,66731a400345e149c87d17a7,6864293a4211dc655073d54c,6864293a4211dc655073d54d,NaN,NaN,NaN,2026-04-18 13:00:08.737143 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True
freq,1,1,3,729,1129,1358,NaN,774,1387,497,...,1,1388,1,1,NaN,NaN,NaN,1,1351,729
mean,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN


### Diagnóstico de Qualidade

Antes de qualquer tratamento, o projeto mede a presença de valores faltantes, duplicidades e possíveis sinais de inconsistência. Esse diagnóstico orienta o tipo de limpeza adotado e evita decisões automáticas sem fundamento no contexto do negócio.


In [4]:
print('Nulos nas movimentações:')
display(df_mov.isna().sum().sort_values(ascending=False).head(10))

print('Nulos nos produtos/serviÃ§os:')
display(df_prod.isna().sum().sort_values(ascending=False).head(10))

print('Duplicadas nas movimentações:', df_mov.duplicated().sum())
print('Duplicadas nos produtos/serviÃ§os:', df_prod.duplicated(subset=['id_produto_servico']).sum())


Nulos nas movimentações:


data_emissao                   0
status_item_cancelado          0
valor_unitario                 0
qtd_venda                      0
qtd_item_movimentacao          0
descricao_cfop                 0
cd_cfop                        0
valor_frete_item               0
valor_desconto_proporcional    0
valor_desconto_digitado        0
dtype: int64

Nulos nos produtos/serviÃ§os:


codigo_barras_tributavel            1302
codigo_barras                        789
vendavel                              29
id_estoque_referencia                  2
id_empresa_referencia                  1
pesavel                                1
percentual_comissao_referencia         1
limite_desconto_referencia             1
margem_lucro_aplicada_referencia       1
id_preco_referencia                    1
dtype: int64

Duplicadas nas movimentações: 8
Duplicadas nos produtos/serviÃ§os: 0


### Limpeza e Integração

O projeto converte os campos para os tipos corretos, integra as fontes e filtra apenas o que representa, de fato, transações de venda com saída de estoque. Essa filtragem reduz ruído, melhora a consistência e mantém o foco no caso de negócio.


In [5]:
df_mov = cast_types_movimentacoes(df_mov)
df_prod = cast_types_produtos(df_prod)
df_base = merge_base(df_mov, df_prod)

for coluna in ['descricao', 'descricao_modelo', 'descricao_situacao', 'descricao_cfop', 'tipo_item_descricao', 'sub_grupo_referencia', 'unidade_sigla']:
    if coluna in df_base.columns:
        df_base[coluna] = df_base[coluna].fillna('desconhecido')

df_base = clean_base(df_base)

print('Base após limpeza:', df_base.shape)
print('Período da base limpa:', df_base['data_emissao'].min(), 'até', df_base['data_emissao'].max())
display(df_base.head())


Base após limpeza: (667, 42)
Período da base limpa: 2026-05-01 00:00:00 até 2026-05-12 00:00:00


,data_emissao,id_produto_servico,id_produto_servico_empresa,tipo_transacao,direcao_estoque,cd_modelo,descricao_modelo,cd_modelo_fiscal,id_pessoa,cd_situacao,...,id_produto_servico_empresa_referencia,id_empresa_referencia,id_estoque_referencia,id_preco_referencia,margem_lucro_aplicada_referencia,limite_desconto_referencia,percentual_comissao_referencia,_source_ingestion_timestamp,_source_file,flag_item_ativo
0,2026-05-01,662875b00345e118a4611764,66731dde0345e118a4611790,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,66731ddf0345e118a46131bd,2,...,66731dde0345e118a4611790,66731a400345e149c87d17a7,66731dde0345e118a4611766,66731dde0345e118a461177a,0.0,0.0,0.0,2026-04-18 13:00:08.529953 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True
1,2026-05-01,68adf63a4211dc2f3475d917,68adf63a4211dc2f3475d91c,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,66731ddf0345e118a46131bd,2,...,68adf63a4211dc2f3475d91c,66731a400345e149c87d17a7,68adf63a4211dc2f3475d918,68adf63a4211dc2f3475d919,0.0,0.0,0.0,2026-04-18 13:00:08.780070 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True
2,2026-05-01,6052c2300345e118a461223e,66731dde0345e118a4612268,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,69e3e0234211dc2e8c649ed4,2,...,66731dde0345e118a4612268,66731a400345e149c87d17a7,66731dde0345e118a461224f,66731dde0345e118a4612265,0.0,0.0,0.0,2026-04-18 13:00:08.438872 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True
3,2026-05-01,67e456138df90b3394089f54,67e456138df90b3394089f59,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,69e3e0234211dc2e8c649ed4,2,...,67e456138df90b3394089f59,66731a400345e149c87d17a7,67e456138df90b3394089f55,67e456138df90b3394089f56,0.0,0.0,0.0,2026-04-18 13:00:08.689883 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True
4,2026-05-01,686428e94211dc655073d519,686428e94211dc655073d51e,VENDA,SAIDA,24,Nota Fiscal Consumidor Eletrônica,65,69e3e0234211dc2e8c649ed4,2,...,686428e94211dc655073d51e,66731a400345e149c87d17a7,686428e94211dc655073d51a,686428e94211dc655073d51b,0.0,0.0,0.0,2026-04-18 13:00:08.736837 UTC,produtos_servicos_2026-04-18_10-00-08.ndjson,True


### Tratamento e Verificação Final

Depois da limpeza, a base já está adequada para a etapa analítica. Aqui o projeto cria variáveis auxiliares de contexto, consolida os dados e realiza uma última checagem de consistência antes de salvar o resultado final.


In [6]:
df_base['receita_bruta_item'] = df_base['qtd_venda'] * df_base['valor_unitario']
df_base['desconto_total_item'] = df_base[['valor_desconto_digitado', 'valor_desconto_proporcional']].fillna(0).sum(axis=1)
df_base['receita_liquida_item'] = df_base['receita_bruta_item'] - df_base['desconto_total_item'] + df_base['valor_frete_item'].fillna(0)
q1 = df_base['receita_liquida_item'].quantile(0.25)
q3 = df_base['receita_liquida_item'].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
df_base['flag_outlier_receita'] = ~df_base['receita_liquida_item'].between(limite_inferior, limite_superior)
df_base['dia_semana'] = df_base['data_emissao'].dt.dayofweek.map({0:'segunda',1:'terca',2:'quarta',3:'quinta',4:'sexta',5:'sabado',6:'domingo'})
df_base['mes_referencia'] = df_base['data_emissao'].dt.strftime('%Y-%m')
df_base['eh_final_de_semana'] = df_base['dia_semana'].isin(['sabado', 'domingo'])

resumo_base = pd.DataFrame({
    'Métrica': [
        'Registros brutos nas movimentações',
        'Registros após a limpeza',
        'Período analisado',
        'Produtos distintos',
        'Outliers de receita',
        'Itens com desconto',
    ],
    'Valor': [
        len(df_mov),
        len(df_base),
        f"{df_base['data_emissao'].min().date()} até {df_base['data_emissao'].max().date()}",
        df_base['id_produto_servico'].nunique(),
        int(df_base['flag_outlier_receita'].sum()),
        int((df_base['desconto_total_item'] > 0).sum()),
    ]
})

display(resumo_base)
print('Base tratada salva em preparação para a Parte 2.')
saida_limpeza = PROCESSED_DIR / 'base_vendas_limpa.csv'
df_base.to_csv(saida_limpeza, index=False)
print('Base limpa salva em:', saida_limpeza)


,Métrica,Valor
0,Registros brutos nas movimentações,708
1,Registros após a limpeza,667
2,Período analisado,2026-05-01 até 2026-05-12
3,Produtos distintos,143
4,Outliers de receita,46
5,Itens com desconto,109


Base tratada salva em preparação para a Parte 2.
Base limpa salva em: /workspace/dados/processed/base_vendas_limpa.csv


## Conclusão

Nesta primeira parte, o projeto transforma os dados brutos em uma base limpa, consistente e pronta para a etapa analítica seguinte. O resultado reforça um princípio central de projetos em ciência de dados em nível de pós-graduação: a qualidade da base precisa ser validada antes de qualquer interpretação mais sofisticada.


### Conexão com a Parte 2

A base tratada gerada aqui será reutilizada no notebook seguinte. Isso garante continuidade entre as etapas e evita retrabalho, além de mostrar que o pré-processamento não é um exercício isolado, mas uma etapa fundadora da análise.


### Leitura de Qualidade e Escopo

O diagnóstico mostrou um conjunto de dados com boa consistência estrutural para o problema proposto. A limpeza preservou **667 registros válidos** e o recorte final ficou concentrado em **11 dias de movimento**, cobrindo o intervalo de **2026-05-01 a 2026-05-12**.

Um ponto importante para a interpretação é que a base final passou a representar **apenas vendas com saída de estoque**, o que evita misturar comportamento comercial com eventos operacionais que não ajudam a responder à pergunta do negócio.

Além disso, a variedade de produtos encontrados é relevante: a base final contém **143 produtos distintos**, indicando um mix com diversidade suficiente para análises posteriores de concentração de receita, sazonalidade e dependência de itens específicos.
